<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/04-pythonic-tools/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Pythonic Tools

This module covers the features that make Python code short, fast and safe once you know them: iterators and generators, `itertools`, decorators, context managers and type hints.

You've already used all of them without necessarily noticing. Every `for` loop uses the iterator protocol. The `DataLoader` in Module 3 used `yield`. `@dataclass` and `@property` are decorators. `with open(...)` is a context manager. And in ML code you see them constantly: `@torch.no_grad()`, `with torch.autocast(...)`, `@functools.lru_cache`, model registries built with decorators, and data pipelines built from generators.

**Contents**

1. How iteration really works
2. Generators
3. Generator pipelines: processing data that doesn't fit in memory
4. `itertools`
5. Decorators
6. Useful decorators from `functools`
7. Context managers
8. Type hints
9. Putting it together: a streaming training pipeline

## 1. How iteration really works

What does `for x in something` actually do? Python asks `something` for an **iterator** by calling `iter(something)`, then calls `next()` on that iterator over and over until it raises `StopIteration`. You can do the same thing by hand:

In [1]:
losses = [0.9, 0.6, 0.4]

it = iter(losses)
print(next(it))
print(next(it))
print(next(it))

try:
    next(it)
except StopIteration:
    print("StopIteration: the loop would end here")

0.9
0.6
0.4
StopIteration: the loop would end here


So a `for` loop is really:

```python
it = iter(losses)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    ...  # loop body
```

There are two separate ideas here:

- An **iterable** is anything you can loop over: lists, strings, dicts, files, ranges. It has an `__iter__` method that gives you a fresh iterator.
- An **iterator** is the object that keeps track of where you are. It has a `__next__` method that returns the next item. Iterators are single-use: once they're used up, they stay used up.

The single-use part catches people out, because several built-ins return iterators rather than lists:

In [2]:
pairs = zip([1, 2, 3], ["a", "b", "c"])

print(list(pairs))
print(list(pairs))    # empty: the zip iterator was already used up

[(1, 'a'), (2, 'b'), (3, 'c')]
[]


`zip`, `map`, `filter`, `enumerate`, `reversed` and file objects all work like this. If you need to go over the result twice, turn it into a list first. A common version of this bug: computing a statistic over `map(...)` once, then looping over the same `map` object again for training, and getting zero iterations without any error.

### Writing your own iterator

A class becomes an iterator by implementing `__iter__` (returning itself) and `__next__`:

In [3]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


for n in Countdown(3):
    print(n)

3
2
1


That works, but it's a lot of boilerplate for "count down from 3". You have to store the state on `self` and remember to raise `StopIteration`. Generators make all of this much easier.

## 2. Generators

A **generator function** is any function that contains `yield`. Calling it doesn't run the body. Instead, it returns a generator object, which is an iterator. Each `next()` runs the body until the next `yield`, hands out that value, and **pauses** there, with all its local variables intact, until the next `next()`.

In [4]:
def countdown(start):
    print("  (starting)")
    current = start
    while current > 0:
        yield current
        current -= 1
    print("  (finished)")


gen = countdown(3)
print(gen)              # nothing has run yet
print(next(gen))
print(next(gen))
print(next(gen))
print(list(gen))        # runs to the end: prints "finished", yields nothing more

<generator object countdown at 0x7fd8d01275e0>
  (starting)
3
2
1
  (finished)
[]


Same behaviour as the `Countdown` class, in a third of the code. The pausing is what makes it work: the local variable `current` survives between calls, so the generator remembers where it was without you storing anything on `self`.

### Why this matters: memory

A list holds every item in memory at once. A generator produces one item at a time and holds only what it needs to produce the next one.

In [5]:
import sys

squares_list = [i * i for i in range(1_000_000)]
squares_gen = (i * i for i in range(1_000_000))

print(f"list:      {sys.getsizeof(squares_list):>12,} bytes")
print(f"generator: {sys.getsizeof(squares_gen):>12,} bytes")

print(sum(squares_gen))

list:         8,448,728 bytes
generator:          200 bytes
333332833333500000


`sys.getsizeof` only measures the container itself (the list's real memory use is even higher, since every integer is a separate object), but the point is clear: the generator's size doesn't depend on how many items it produces. A generator over a billion items takes the same tiny amount of memory as one over ten.

That's why data loaders are built on generators. A dataset of 100 GB of images can't be loaded into a list, but a generator can read one batch at a time, forever if needed.

### Infinite generators

Since a generator only produces values when asked, it can go on forever. That's useful for things like an endless stream of training batches, where the training loop decides when to stop:

In [6]:
def cycle_batches(data, batch_size):
    """Yield batches forever, starting over at the end of the data."""
    while True:
        for start in range(0, len(data), batch_size):
            yield data[start:start + batch_size]


stream = cycle_batches(list(range(7)), batch_size=3)
for step in range(5):
    print(f"step {step}: {next(stream)}")

step 0: [0, 1, 2]
step 1: [3, 4, 5]
step 2: [6]
step 3: [0, 1, 2]
step 4: [3, 4, 5]


Many large-scale training scripts are organised around a fixed number of *steps* rather than epochs, and an endless batch stream like this is exactly how they do it.

### yield from

`yield from other` yields every item from another iterable. It's how you delegate part of a generator to another one:

In [7]:
def all_files(train_files, val_files):
    yield from train_files
    yield from val_files


print(list(all_files(["a.jpg", "b.jpg"], ["c.jpg"])))

['a.jpg', 'b.jpg', 'c.jpg']


### Generator expressions

You met these in Module 1: a comprehension with parentheses instead of brackets. They're a quick way to write a simple generator inline:

In [8]:
tokens = ["the", "cat", "sat", "on", "the", "mat"]
lengths = (len(t) for t in tokens)
print(max(lengths))

3


Use a generator expression for a simple one-line transformation, and a generator function when you need loops, conditions or state.

## 3. Generator pipelines

Generators really shine when you chain them. Each stage takes an iterator in and yields items out, and data flows through the whole pipeline one item at a time. Nothing is ever loaded all at once.

Let's process a log of training examples. In real life this would be a large file on disk (Module 5 covers reading files). Here a generator stands in for it, producing lines on demand, including some junk lines that need to be cleaned out:

In [9]:
import random


def read_lines(n, seed=0):
    """Stand-in for reading a big file line by line."""
    rng = random.Random(seed)
    for i in range(n):
        if i % 50 == 0:
            yield "# checkpoint comment"
        elif i % 37 == 0:
            yield "corrupted,,line"
        else:
            yield f"{i},{rng.uniform(0, 10):.3f},{rng.choice(['cat', 'dog'])}"


for line in list(read_lines(5)):
    print(line)

# checkpoint comment
1,8.444,dog
2,0.405,dog
3,4.049,dog
4,9.678,dog


Now the pipeline stages, each one small and testable on its own:

In [10]:
def parse(lines):
    for line in lines:
        if line.startswith("#"):
            continue
        parts = line.split(",")
        try:
            yield {"id": int(parts[0]), "value": float(parts[1]), "label": parts[2]}
        except (ValueError, IndexError):
            continue            # skip malformed lines


def only_label(records, label):
    for r in records:
        if r["label"] == label:
            yield r


def batches(items, size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

And connecting them is just passing one generator into the next:

In [11]:
lines = read_lines(1_000_000)
records = parse(lines)
cats = only_label(records, "cat")
cat_batches = batches(cats, 4)

print(next(cat_batches))
print(next(cat_batches))

[{'id': 5, 'value': 5.834, 'label': 'cat'}, {'id': 7, 'value': 1.397, 'label': 'cat'}, {'id': 9, 'value': 9.873, 'label': 'cat'}, {'id': 10, 'value': 3.101, 'label': 'cat'}]
[{'id': 12, 'value': 4.721, 'label': 'cat'}, {'id': 14, 'value': 6.109, 'label': 'cat'}, {'id': 17, 'value': 0.623, 'label': 'cat'}, {'id': 19, 'value': 7.103, 'label': 'cat'}]


We built a pipeline over a million lines and pulled two batches out of it instantly. Only about twenty lines have actually been read so far. The rest of the million haven't been generated, parsed or filtered yet, and they won't be unless something asks for them. This is called **lazy evaluation**.

If we wanted to process everything, we'd just keep looping, and memory use would stay flat the whole time:

In [12]:
total = 0
count = 0
for record in only_label(parse(read_lines(200_000)), "dog"):
    total += record["value"]
    count += 1

print(f"{count:,} dogs, mean value {total / count:.3f}")

95,507 dogs, mean value 4.999


This shape (read, parse, filter, transform, batch) is the shape of almost every data pipeline, from `tf.data` to Hugging Face's streaming datasets. The libraries add parallelism and caching on top, but the core idea is a chain of lazy iterators.

## 4. itertools

The `itertools` module is a toolbox of fast iterator building blocks. Here are the ones that come up most in ML code.

### islice: take part of an iterator

You can't slice a generator with `[:5]`, since it has no length or indexes. `islice` does the same thing lazily:

In [13]:
import itertools

stream = cycle_batches(list(range(10)), batch_size=4)
for batch in itertools.islice(stream, 4):
    print(batch)

[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9]
[0, 1, 2, 3]


### count, cycle, repeat: infinite iterators

In [14]:
for step, lr in zip(itertools.count(start=1), [0.1, 0.05, 0.01]):
    print(step, lr)

colors = itertools.cycle(["red", "green", "blue"])
print([next(colors) for _ in range(5)])

1 0.1
2 0.05
3 0.01
['red', 'green', 'blue', 'red', 'green']


### chain: one iterator after another

In [15]:
train_files = ["a.jpg", "b.jpg"]
extra_files = ["c.jpg"]
print(list(itertools.chain(train_files, extra_files)))

nested = [[1, 2], [3], [4, 5, 6]]
print(list(itertools.chain.from_iterable(nested)))     # flatten one level

['a.jpg', 'b.jpg', 'c.jpg']
[1, 2, 3, 4, 5, 6]


### product: every combination

`product` gives every combination of items from several iterables, which is exactly a hyperparameter grid:

In [16]:
grid = {
    "lr": [0.1, 0.01],
    "batch_size": [16, 32],
    "dropout": [0.0, 0.3],
}

for values in itertools.product(*grid.values()):
    config = dict(zip(grid.keys(), values))
    print(config)

{'lr': 0.1, 'batch_size': 16, 'dropout': 0.0}
{'lr': 0.1, 'batch_size': 16, 'dropout': 0.3}
{'lr': 0.1, 'batch_size': 32, 'dropout': 0.0}
{'lr': 0.1, 'batch_size': 32, 'dropout': 0.3}
{'lr': 0.01, 'batch_size': 16, 'dropout': 0.0}
{'lr': 0.01, 'batch_size': 16, 'dropout': 0.3}
{'lr': 0.01, 'batch_size': 32, 'dropout': 0.0}
{'lr': 0.01, 'batch_size': 32, 'dropout': 0.3}


`*grid.values()` unpacks the three lists into three arguments, and `dict(zip(keys, values))` turns each combination back into a config dict. You'll use this in the mini project to build a grid search.

### combinations and permutations

In [17]:
models = ["knn", "logreg", "tree"]
print(list(itertools.combinations(models, 2)))    # every pair, order doesn't matter

[('knn', 'logreg'), ('knn', 'tree'), ('logreg', 'tree')]


Useful for things like comparing every pair of models, or picking which features to combine.

### accumulate: running totals

In [18]:
batch_losses = [2.0, 1.5, 1.2, 0.9]
print(list(itertools.accumulate(batch_losses)))           # running sum
print(list(itertools.accumulate(batch_losses, min)))      # best so far

[2.0, 3.5, 4.7, 5.6000000000000005]
[2.0, 1.5, 1.2, 0.9]


### groupby: group consecutive items

`groupby` groups **consecutive** items that share a key. That word matters: it doesn't sort for you, so items with the same key that aren't next to each other end up in separate groups.

In [19]:
records = [("cat", 1), ("dog", 2), ("cat", 3), ("cat", 4)]

print("unsorted:")
for label, group in itertools.groupby(records, key=lambda r: r[0]):
    print(" ", label, list(group))

print("sorted first:")
for label, group in itertools.groupby(sorted(records), key=lambda r: r[0]):
    print(" ", label, list(group))

unsorted:
  cat [('cat', 1)]
  dog [('dog', 2)]
  cat [('cat', 3), ('cat', 4)]
sorted first:
  cat [('cat', 1), ('cat', 3), ('cat', 4)]
  dog [('dog', 2)]


If your data isn't already sorted by the key, either sort it first or use a `defaultdict(list)` from Module 2, which doesn't care about order.

### batched (Python 3.12+)

Newer versions of Python have `itertools.batched`, which does what our `batches` function did:

In [20]:
if hasattr(itertools, "batched"):
    print(list(itertools.batched(range(7), 3)))
else:
    print("itertools.batched needs Python 3.12 or newer")

[(0, 1, 2), (3, 4, 5), (6,)]


Note it yields tuples, not lists. You'll write your own version in the exercises, which works on any Python version.

## 5. Decorators

A decorator is a function that takes a function and returns a new function, usually one that wraps the original with some extra behaviour. You've used them already: `@property`, `@dataclass`, `@staticmethod`.

To see how they work, we start with two facts from Module 1: functions are objects you can pass around, and a function defined inside another function can use the outer function's variables (a closure, as in the Module 3 mini project).

Here's a function that takes a function and returns a timed version of it:

In [21]:
import time


def timed(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed * 1000:.1f} ms")
        return result
    return wrapper


def slow_sum(n):
    return sum(i * i for i in range(n))


slow_sum = timed(slow_sum)       # replace it with the wrapped version
print(slow_sum(1_000_000))

slow_sum took 48.9 ms
333332833333500000


`wrapper` takes `*args, **kwargs` so it can pass along whatever arguments the original function needs, without knowing what they are. It calls the original, does its extra work, and returns the original's result.

The line `slow_sum = timed(slow_sum)` is what a decorator does. The `@` syntax is just a shorter way of writing it:

In [22]:
@timed
def count_multiples(n, k):
    return sum(1 for i in range(n) if i % k == 0)


print(count_multiples(1_000_000, 7))

count_multiples took 37.4 ms
142858


`@timed` above the `def` means exactly "define `count_multiples`, then replace it with `timed(count_multiples)`".

### functools.wraps

There's a problem with our decorator. The wrapped function has lost its identity:

In [23]:
print(count_multiples.__name__)
print(count_multiples.__doc__)

wrapper
None


It reports its name as `wrapper`, so error messages, logs and `help()` all show the wrong thing. `functools.wraps` fixes this by copying the original's name, docstring and other details onto the wrapper. Always use it:

In [24]:
import functools


def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {(time.perf_counter() - start) * 1000:.1f} ms")
        return result
    return wrapper


@timed
def normalize(values):
    """Scale values to the range 0 to 1."""
    lo, hi = min(values), max(values)
    return [(v - lo) / (hi - lo) for v in values]


print(normalize.__name__, "|", normalize.__doc__)
print(normalize([1, 5, 9]))

normalize | Scale values to the range 0 to 1.
normalize took 0.0 ms
[0.0, 0.5, 1.0]


### Decorators with arguments

What if you want to configure the decorator, as in `@retry(times=3)`? Then you need one more layer: a function that takes the settings and *returns a decorator*.

In [25]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as e:
                    print(f"  attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator


calls = {"n": 0}

@retry(times=3)
def download_weights():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("server busy")
    return "weights.bin"


print(download_weights())

  attempt 1 failed: server busy
  attempt 2 failed: server busy
weights.bin


Three levels of functions looks confusing at first. Read it from the outside in: `retry(times=3)` runs first and returns `decorator`. Then `@decorator` is applied to `download_weights`, which returns `wrapper`. The `times` value is remembered by the closures.

Retrying on network errors is a real need when downloading datasets or model weights, or calling an API. You'll write a more complete version in the exercises.

### A registry decorator

A decorator doesn't have to wrap the function at all. It can just record it somewhere and hand it back unchanged. That's how many ML libraries let you refer to models or functions by name, in a config file for example:

In [26]:
ACTIVATIONS = {}


def register(name):
    def decorator(func):
        ACTIVATIONS[name] = func
        return func             # unchanged
    return decorator


@register("relu")
def relu(x):
    return max(0.0, x)


@register("leaky_relu")
def leaky_relu(x):
    return x if x > 0 else 0.01 * x


config = {"activation": "leaky_relu"}
act = ACTIVATIONS[config["activation"]]
print(sorted(ACTIVATIONS), act(-2.0))

['leaky_relu', 'relu'] -0.02


Libraries like timm, Hugging Face and Detectron2 all use registries like this. Adding a new model means writing a class and putting `@register("my_model")` above it, and no other code has to change. The mini project uses this pattern.

## 6. Useful decorators from functools

### lru_cache and cache: remember results

`functools.lru_cache` stores the results of a function, keyed by its arguments. If it's called again with the same arguments, it returns the stored result instead of recomputing it. This is called **memoization**.

In [27]:
@functools.lru_cache(maxsize=None)
def load_dataset(name):
    print(f"  (loading {name} from disk...)")
    time.sleep(0.2)
    return list(range(1000))


for _ in range(3):
    data = load_dataset("train")
print(len(data))
print(load_dataset.cache_info())

  (loading train from disk...)


1000
CacheInfo(hits=2, misses=1, maxsize=None, currsize=1)


The expensive load only happened once. `maxsize` limits how many results to keep; when it's full, the least recently used one is thrown away (that's what LRU means). `functools.cache` is the same thing with no limit.

There are two important limits:

1. **The arguments must be hashable**, since they're used as dictionary keys. You can't cache a function that takes a list.
2. **Only cache functions whose result depends only on their arguments.** If `load_dataset` read a file that has changed since, the cache would happily return the old data.

The classic demonstration is a recursive function that recomputes the same values over and over:

In [28]:
def fib_slow(n):
    return n if n < 2 else fib_slow(n - 1) + fib_slow(n - 2)


@functools.cache
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)


start = time.perf_counter()
print(fib_slow(27), f"{time.perf_counter() - start:.3f} s")

start = time.perf_counter()
print(fib_fast(27), f"{time.perf_counter() - start:.6f} s")
print(fib_fast(300))

196418 0.027 s
196418 0.000064 s
222232244629420445529739893461909967206666939096499764990979600


`fib_slow(27)` makes about 600,000 function calls. `fib_fast(27)` makes 28. Dynamic programming algorithms (like the Viterbi algorithm used in speech recognition and sequence tagging) are built on this same idea of reusing results instead of recomputing them.

### partial: fix some arguments in advance

`functools.partial` creates a new function with some arguments already filled in:

In [29]:
def scaled_loss(pred, target, weight):
    return weight * (pred - target) ** 2


heavy_loss = functools.partial(scaled_loss, weight=10.0)
print(heavy_loss(0.5, 1.0))

int_from_binary = functools.partial(int, base=2)
print(int_from_binary("1011"))

2.5
11


This is handy when a library expects a function with a certain signature, like a loss function that takes `(pred, target)`, and yours needs an extra setting.

## 7. Context managers

A context manager sets something up when a `with` block starts and guarantees it's cleaned up when the block ends, **even if an error happens inside**. The classic example is files:

```python
with open("results.txt", "w") as f:
    f.write("accuracy: 0.91")
# the file is closed here, even if write() raised an error
```

Without `with`, if an error happens between opening and closing, the file stays open. The same goes for locks, database connections, GPU memory, temporary settings and anything else that must be undone.

### Writing one as a class

The `with` statement calls `__enter__` at the start and `__exit__` at the end. Whatever `__enter__` returns is what you get after `as`.

In [30]:
class Timer:
    def __init__(self, name="block"):
        self.name = name
        self.elapsed = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self._start
        print(f"{self.name}: {self.elapsed * 1000:.1f} ms")
        return False      # don't swallow exceptions


with Timer("sorting") as t:
    sorted(random.random() for _ in range(200_000))

print(f"stored: {t.elapsed:.4f} s")

sorting: 57.0 ms
stored: 0.0570 s


`__exit__` receives information about any exception that happened in the block (all three are `None` if nothing went wrong). If it returns `True`, the exception is suppressed. Returning `False` lets it continue as normal, which is almost always what you want. A context manager that silently swallows errors hides bugs.

In [31]:
try:
    with Timer("failing block"):
        raise ValueError("something broke")
except ValueError as e:
    print("caught:", e)

failing block: 0.0 ms
caught: something broke


The timer still printed, even though the block crashed. That's the guarantee.

### Writing one with contextlib

For simple cases, `contextlib.contextmanager` lets you write a context manager as a generator. Everything before `yield` is the setup, everything after it is the cleanup, and wrapping the `yield` in `try/finally` makes sure the cleanup runs even on errors:

In [32]:
from contextlib import contextmanager


@contextmanager
def temporary_seed(seed):
    """Run a block with a fixed random seed, then restore the previous random state."""
    state = random.getstate()
    random.seed(seed)
    try:
        yield
    finally:
        random.setstate(state)


random.seed(123)
with temporary_seed(0):
    a = [random.randint(0, 9) for _ in range(5)]
with temporary_seed(0):
    b = [random.randint(0, 9) for _ in range(5)]

print(a, b, a == b)

[6, 6, 0, 4, 8] [6, 6, 0, 4, 8] True


Both blocks produced the same numbers, and the global random state was put back afterwards, so the rest of the program isn't affected. Reproducible evaluation code often does exactly this.

### The ML connection

PyTorch's `torch.no_grad()` is a context manager. Inside it, operations don't build the autograd graph from the Module 3 mini project, which saves memory during evaluation. Here's the same idea, applied to a simple global setting:

In [33]:
SETTINGS = {"track_gradients": True}


@contextmanager
def no_grad():
    previous = SETTINGS["track_gradients"]
    SETTINGS["track_gradients"] = False
    try:
        yield
    finally:
        SETTINGS["track_gradients"] = previous


print(SETTINGS)
with no_grad():
    print(SETTINGS)
print(SETTINGS)

{'track_gradients': True}
{'track_gradients': False}
{'track_gradients': True}


Saving the previous value, rather than setting it back to `True`, means it also works correctly when nested inside another `no_grad()` block. The `torch.autocast` (mixed precision) and `model.eval()` patterns are built on the same idea: change a setting temporarily and always put it back.

## 8. Type hints

Type hints describe what types a function expects and returns:

In [34]:
def accuracy(y_true: list[int], y_pred: list[int]) -> float:
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


print(accuracy([1, 0, 1], [1, 1, 1]))
print(accuracy.__annotations__)

0.6666666666666666
{'y_true': list[int], 'y_pred': list[int], 'return': <class 'float'>}


Python **does not check them** when the code runs. `accuracy("abc", "abd")` would still run. They're documentation that tools can read: your editor uses them for autocompletion and warnings, and a type checker like `mypy` can scan a whole project and find type mistakes before you run anything.

So why bother? In a large ML codebase, the hardest bugs are often about shapes and types: a function that expects a list of floats gets a list of lists, or a config value is a string instead of a number. Type hints make those expectations visible, and they make code much easier to read for the next person, who is often you in three months.

### The common forms

In [35]:
from typing import Callable, Iterator, Optional


def mean(values: list[float]) -> float:
    return sum(values) / len(values)


def class_counts(labels: list[str]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return counts


def load(path: str, limit: int | None = None) -> list[str]:     # same as Optional[int]
    ...


def evaluate(metric: Callable[[list[int], list[int]], float]) -> float:
    return metric([1, 0], [1, 1])


def read_batches(n: int) -> Iterator[list[int]]:
    for i in range(n):
        yield [i, i + 1]


print(evaluate(accuracy))

0.5


- `list[float]`, `dict[str, int]`, `tuple[int, int]` describe containers and what's in them.
- `int | None` means "an int, or None". `Optional[int]` is the older way to write the same thing.
- `Callable[[arg types], return type]` describes a function.
- `Iterator[X]` is the return type of a generator that yields `X`.

### Protocols: types for duck typing

In Module 3, the `Pipeline` worked with any object that had `fit` and `transform`, with no shared base class required. `typing.Protocol` lets you write that expectation down as a type:

In [36]:
from typing import Protocol


class Classifier(Protocol):
    def fit(self, X: list[list[float]], y: list[int]) -> "Classifier": ...
    def predict(self, X: list[list[float]]) -> list[int]: ...


def cross_check(model: Classifier, X: list[list[float]], y: list[int]) -> float:
    model.fit(X, y)
    return accuracy(y, model.predict(X))

Any class with matching `fit` and `predict` methods counts as a `Classifier` for the type checker, without inheriting from anything. It describes the interface without forcing a class hierarchy on anyone.

How much to annotate is a judgement call. A good default: annotate function signatures (arguments and return values) in any code that's reused or shared, and don't bother with local variables unless the type isn't obvious. The rest of this module's notebooks follow that rule.

## 9. Putting it together: a streaming training pipeline

Let's combine everything into a training setup that never holds the whole dataset in memory:

- a **generator** that streams examples, standing in for reading a huge file
- **generator stages** to clean and batch them
- **`itertools.islice`** to train for a fixed number of steps
- a **decorator** that logs each training step's duration
- a **context manager** that times the whole run
- **type hints** on every function

The model is a simple linear regression trained with gradient descent, as in Module 1, but now it learns from a stream, one mini-batch at a time.

In [37]:
import math

Example = tuple[float, float]


def stream_examples(seed: int = 0) -> Iterator[Example]:
    """An endless stream of (x, y) pairs where y = 3x - 2 plus noise, with some bad rows."""
    rng = random.Random(seed)
    while True:
        x = rng.uniform(-1, 1)
        y = 3 * x - 2 + rng.gauss(0, 0.1)
        if rng.random() < 0.02:
            y = float("nan")              # simulate a corrupted row
        yield x, y


def drop_invalid(examples: Iterator[Example]) -> Iterator[Example]:
    for x, y in examples:
        if not (math.isnan(x) or math.isnan(y)):
            yield x, y


def batch_stream(examples: Iterator[Example], size: int) -> Iterator[list[Example]]:
    while True:
        batch = list(itertools.islice(examples, size))
        if not batch:
            return
        yield batch

`batch_stream` uses a neat trick: `islice(examples, size)` takes the next `size` items from the *same* iterator each time, because an iterator remembers where it is. There's no counter to manage.

Next, a decorator that keeps track of how long each step takes, and the training step itself:

In [38]:
def track_time(func: Callable) -> Callable:
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        wrapper.total_time += time.perf_counter() - start
        wrapper.calls += 1
        return result
    wrapper.total_time = 0.0
    wrapper.calls = 0
    return wrapper


@track_time
def train_step(params: dict[str, float], batch: list[Example], lr: float) -> float:
    n = len(batch)
    errors = [(params["w"] * x + params["b"]) - y for x, y in batch]
    params["w"] -= lr * 2 / n * sum(e * x for e, (x, _) in zip(errors, batch))
    params["b"] -= lr * 2 / n * sum(errors)
    return sum(e * e for e in errors) / n

The decorator stores its statistics as attributes on the wrapper function itself. Functions are objects, so you can attach attributes to them like any other object.

Now the run:

In [39]:
params = {"w": 0.0, "b": 0.0}
train_batches = batch_stream(drop_invalid(stream_examples(seed=1)), size=32)

with Timer("training") as run:
    for step, batch in enumerate(itertools.islice(train_batches, 300)):
        loss = train_step(params, batch, lr=0.1)
        if step % 50 == 0:
            print(f"step {step:3d}  loss={loss:.4f}  w={params['w']:.3f}  b={params['b']:.3f}")

print(f"\nlearned w={params['w']:.3f}, b={params['b']:.3f}  (true: w=3, b=-2)")
print(f"{train_step.calls} steps, average {train_step.total_time / train_step.calls * 1e6:.0f} µs per step")

step   0  loss=5.8613  w=0.168  b=-0.327
step  50  loss=0.0095  w=2.912  b=-1.993
step 100  loss=0.0121  w=2.998  b=-1.995
step 150  loss=0.0083  w=3.001  b=-2.008
step 200  loss=0.0068  w=3.009  b=-2.008
step 250  loss=0.0115  w=3.001  b=-1.997
training: 16.7 ms

learned w=2.998, b=-2.007  (true: w=3, b=-2)
300 steps, average 14 µs per step


The model recovered the true line, the corrupted rows were filtered out on the way, and the dataset was infinite, so it could never have fit in a list. At no point were more than 32 examples in memory at once.

Every piece is small and does one job: the source doesn't know about cleaning, the cleaner doesn't know about batching, and the training loop doesn't know where the data comes from. Swap the fake stream for a real file reader and nothing else needs to change.

## Summary

- A `for` loop calls `iter()` once and `next()` until `StopIteration`. Iterators are single-use: `zip`, `map`, `filter` and files can only be looped over once.
- A function with `yield` is a generator. It runs lazily, pauses at each `yield` and keeps its local variables. Its memory use doesn't grow with the number of items.
- Chain generators into pipelines (read, parse, filter, batch) to process data that doesn't fit in memory.
- `itertools`: `islice` to take part of an iterator, `product` for hyperparameter grids, `chain` to join, `accumulate` for running totals, `groupby` only on sorted data.
- A decorator takes a function and returns a new one. Always use `functools.wraps`. Decorators with arguments need one extra layer.
- Registry decorators record a function or class by name and return it unchanged.
- `functools.cache`/`lru_cache` memoize pure functions with hashable arguments. `partial` pre-fills arguments.
- Context managers guarantee cleanup. Write them as a class (`__enter__`/`__exit__`) or with `@contextmanager` and `try/finally`. Don't swallow exceptions.
- Type hints document types for people and tools. Python doesn't enforce them. `Protocol` describes duck-typed interfaces.

Next, open `02_exercises.ipynb`.